# Scholarship Scorer — GPU Backend

Runs the PS + Resume scoring models on a Colab GPU and exposes them via a public URL.

**Before running:**
1. ⚠️ Set runtime to GPU **first**: `Runtime > Change runtime type > T4 GPU` then reconnect
2. Upload `ps_lora_v3_final.zip` and `resume_grader_adapter.zip` to your Google Drive
3. Get a free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ── Step 1: Verify GPU is available ──────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to Runtime > Change runtime type > T4 GPU, then reconnect and run again."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)")
print("CUDA version:", torch.version.cuda)

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok transformers peft bitsandbytes accelerate

In [ ]:
# ── Step 3: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 4: Set adapter zip paths + validate ─────────────────────────────────
# Update these paths to match where YOUR zips are in Google Drive
import os, zipfile

os.environ['PS_ADAPTER_ZIP']     = '/content/drive/MyDrive/ps_lora_v3_final.zip'
os.environ['RESUME_ADAPTER_ZIP'] = '/content/drive/MyDrive/resume_grader_adapter.zip'

for key in ('PS_ADAPTER_ZIP', 'RESUME_ADAPTER_ZIP'):
    p = os.environ[key]

    # Check file exists
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"❌ {key}: file not found at {p}\n"
            f"   Make sure the zip is uploaded to Google Drive at that exact path."
        )

    # Check it is actually a zip (not an HTML error page from Drive)
    if not zipfile.is_zipfile(p):
        size_kb = os.path.getsize(p) / 1024
        raise ValueError(
            f"❌ {key}: {p} is not a valid zip file (size: {size_kb:.1f} KB).\n"
            f"   This usually means the file didn't upload properly.\n"
            f"   Fix: delete it from Google Drive, re-upload the original zip, then run this cell again."
        )

    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f"✅ {key}: valid zip  ({size_mb:.1f} MB)")

In [ ]:
# ── Step 5: Write model.py ───────────────────────────────────────────────────
%%writefile /content/model.py
import json
import os
import re
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_LOADED = False
ps_model = None
ps_tokenizer = None
resume_model = None
resume_tokenizer = None

PS_ADAPTER_ZIP     = Path(os.environ.get('PS_ADAPTER_ZIP',     'ps_lora_v3_final.zip'))
PS_ADAPTER_DIR     = Path(os.environ.get('PS_ADAPTER_DIR',     '/content/adapter'))
PS_BASE_MODEL      = 'unsloth/llama-3-8b-bnb-4bit'

RESUME_ADAPTER_ZIP = Path(os.environ.get('RESUME_ADAPTER_ZIP', 'resume_grader_adapter.zip'))
RESUME_ADAPTER_DIR = Path(os.environ.get('RESUME_ADAPTER_DIR', '/content/adapter_resume'))
RESUME_BASE_MODEL  = 'unsloth/llama-3.2-1b-bnb-4bit'

PS_CRITERIA = ['interests_and_values','academic_commitment','clarity_of_vision','organization','language_quality']
RESUME_CRITERIA = ['academic_achievement','leadership_and_extracurriculars','community_service','research_and_work_experience','skills_and_certifications','awards_and_recognition']

def _extract(zip_path, dest):
    if dest.exists() and any(dest.iterdir()):
        print(f'Adapter already extracted at {dest}')
        return
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dest)
    print(f'Extracted {zip_path.name} -> {dest}')

def _load_peft_model(base_model_id, adapter_dir):
    if not torch.cuda.is_available():
        raise RuntimeError('GPU not available. Cannot load model without CUDA.')
    device_label = torch.cuda.get_device_name(0)
    print(f'  Loading {base_model_id} on {device_label}')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    base = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map='auto')
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval()
    tok = AutoTokenizer.from_pretrained(str(adapter_dir))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model, tok

def load_model():
    global ps_model, ps_tokenizer, resume_model, resume_tokenizer, MODEL_LOADED
    _extract(PS_ADAPTER_ZIP, PS_ADAPTER_DIR)
    _extract(RESUME_ADAPTER_ZIP, RESUME_ADAPTER_DIR)
    print('Loading PS model (LLaMA 3 8B)...')
    ps_model, ps_tokenizer = _load_peft_model(PS_BASE_MODEL, PS_ADAPTER_DIR)
    print('Loading Resume model (LLaMA 3.2 1B)...')
    resume_model, resume_tokenizer = _load_peft_model(RESUME_BASE_MODEL, RESUME_ADAPTER_DIR)
    MODEL_LOADED = True
    print('Both models loaded and ready.')

def _generate(model, tokenizer, prompt, max_new_tokens):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def _extract_json(raw):
    text = re.sub(r'^```(?:json)?\n?', '', raw.strip())
    text = re.sub(r'\n?```$', '', text.strip())
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        raise ValueError(f'No JSON found in model output: {raw!r}')
    return json.loads(match.group())

def score_statement(personal_statement):
    system = ('You are a scholarship evaluation assistant.\n'
              'You evaluate PERSONAL STATEMENTS using a strict rubric and return VALID JSON only.\n'
              'Do not assume missing information. If something is missing, write "Not provided".\n'
              'No text outside JSON. Ensure overall_score equals the sum of criteria scores.')
    user = ('TASK: personal_statement\n\nRubric: Score each criterion 0-20. Total must equal sum.\n\n'
            f'Personal Statement:\n<<<\n{personal_statement}\n>>>')
    prompt = (f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>'
              f'<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>'
              '<|start_header_id|>assistant<|end_header_id|>\n\n')
    raw = _generate(ps_model, ps_tokenizer, prompt, max_new_tokens=512)
    print(f'[PS] Raw output: {raw}')
    result = _extract_json(raw)
    total = sum(int(result.get(k, 0)) for k in PS_CRITERIA)
    result['overall_score'] = total
    result['grade_pct'] = round((total / 100) * 100, 1)
    return result

_RESUME_SYSTEM = """You are a scholarship committee evaluator.
You assess scholarship application RESUMES against a strict 6-criterion rubric.
Each criterion is scored 0-30 (0-10: weak, 11-20: adequate, 21-30: strong).
Return ONLY valid JSON with exactly these keys:
  academic_achievement, leadership_and_extracurriculars, community_service,
  research_and_work_experience, skills_and_certifications, awards_and_recognition,
  overall_score, justification
overall_score MUST equal the sum of the six criteria scores.
Do NOT include any text outside the JSON object."""

_RESUME_USER = """Evaluate the following scholarship application resume using the rubric below.

RUBRIC (each 0-30):
1. academic_achievement          - GPA, honours, academic awards, class rank
2. leadership_and_extracurriculars - club leadership, student orgs, sports, positions
3. community_service             - volunteer hours, social impact, outreach
4. research_and_work_experience  - internships, research, publications, relevant jobs
5. skills_and_certifications     - technical skills, languages, professional certs
6. awards_and_recognition        - scholarships won, competition prizes, recognitions

Scoring bands: 0-10 = not demonstrated / weak | 11-20 = adequate | 21-30 = strong / excellent

RESUME:
<<<
{resume}
>>>

Return valid JSON only. overall_score must equal the sum of all six criteria scores."""

def score_resume(resume_text):
    messages = [
        {'role': 'system', 'content': _RESUME_SYSTEM},
        {'role': 'user',   'content': _RESUME_USER.format(resume=resume_text)},
    ]
    prompt = resume_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    raw = _generate(resume_model, resume_tokenizer, prompt, max_new_tokens=400)
    print(f'[Resume] Raw output: {raw}')
    result = _extract_json(raw)
    total = sum(int(result.get(k, 0)) for k in RESUME_CRITERIA)
    result['overall_score'] = total
    return result

In [ ]:
# ── Step 6: Write main.py ────────────────────────────────────────────────────
%%writefile /content/main.py
import asyncio
from contextlib import asynccontextmanager
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

import model as ml

_executor = ThreadPoolExecutor(max_workers=1)

@asynccontextmanager
async def lifespan(app: FastAPI):
    loop = asyncio.get_event_loop()
    await loop.run_in_executor(_executor, ml.load_model)
    yield

app = FastAPI(title='Scholarship Scorer', lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)

class PSRequest(BaseModel):
    academic_goals: str
    career_goals: str
    leadership_experience: str
    personal_statement: str

class ResumeRequest(BaseModel):
    resume_text: str

@app.get('/health')
def health():
    return {'ok': True, 'model_loaded': ml.MODEL_LOADED}

@app.post('/score/personal-statement')
async def score_ps(req: PSRequest):
    if not ml.MODEL_LOADED:
        raise HTTPException(status_code=503, detail='Model is still loading, try again shortly.')
    try:
        loop = asyncio.get_event_loop()
        scores = await loop.run_in_executor(_executor, ml.score_statement, req.personal_statement)
        return scores
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/score/resume')
async def score_resume(req: ResumeRequest):
    if not ml.MODEL_LOADED:
        raise HTTPException(status_code=503, detail='Model is still loading, try again shortly.')
    try:
        loop = asyncio.get_event_loop()
        scores = await loop.run_in_executor(_executor, ml.score_resume, req.resume_text)
        return scores
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
# ── Step 7: Set ngrok auth token ─────────────────────────────────────────────
# 1. Go to https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Copy your token
# 3. Paste it between the quotes below (replace the placeholder)
from pyngrok import ngrok, conf

NGROK_AUTH_TOKEN = 'PASTE_YOUR_TOKEN_HERE'  # ← replace this

if NGROK_AUTH_TOKEN == 'PASTE_YOUR_TOKEN_HERE' or not NGROK_AUTH_TOKEN.strip():
    raise ValueError(
        "You must set your ngrok auth token!\n"
        "Get it from: https://dashboard.ngrok.com/get-started/your-authtoken"
    )

conf.get_default().auth_token = NGROK_AUTH_TOKEN
print('ngrok auth token set.')

In [ ]:
# ── Step 8: Start the server ─────────────────────────────────────────────────
import threading
import uvicorn
import sys
import time
import requests

sys.path.insert(0, '/content')
ngrok.kill()

server_error = []

def _run():
    try:
        uvicorn.run('main:app', host='0.0.0.0', port=8000, app_dir='/content')
    except Exception as e:
        server_error.append(str(e))

thread = threading.Thread(target=_run, daemon=True)
thread.start()

print('Waiting for models to load (2-5 min on first run)...')
loaded = False
for i in range(120):
    time.sleep(5)

    # Stop early if the server thread died
    if not thread.is_alive():
        raise RuntimeError(
            f"Server crashed on startup. Error: {server_error[0] if server_error else 'unknown'}\n"
            "Scroll up to see the full traceback."
        )

    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.json().get('model_loaded'):
            loaded = True
            print(f'Models loaded after {(i+1)*5}s!')
            break
    except Exception:
        pass

    if (i + 1) % 6 == 0:
        print(f'  Still loading... ({(i+1)*5}s elapsed)')

if not loaded:
    raise RuntimeError('Models did not finish loading within 10 minutes. Check for errors above.')

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

print()
print('=' * 60)
print('Server is live!')
print()
print(f'Public URL: {PUBLIC_URL}')
print()
print('Create frontend/.env with:')
print(f'  VITE_PYTHON_API={PUBLIC_URL}')
print('Then restart: npm run dev')
print('=' * 60)